# AlphaAgentEvo v2 - Full Kaggle Runner

Notebook này có đủ cell để chạy trên Kaggle/VSCode remote:
1. Bootstrap runtime
2. Train + monitor log realtime trong 1 cell
3. Dump debug bundle khi fail

Mặc định dùng model ổn định: `Qwen/Qwen3-0.6B-MLX-bf16`.

In [ ]:
%%bash
set -euo pipefail

REPO="${REPO:-/kaggle/input/datasets/gplebih/aae-new}"
WORK="${WORK:-/kaggle/working/aae_v2}"

[ -d "$REPO/deploy/v2" ] || { echo "ERROR: REPO invalid: $REPO"; exit 1; }

cat > /kaggle/working/aae_env.sh <<EOF
export REPO="$REPO"
export WORK="$WORK"
EOF

echo "REPO=$REPO"
echo "WORK=$WORK"

bash "$REPO/deploy/v2/setup.sh"


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/aae_env.sh

ray stop --force || true
mkdir -p "$WORK/logs"

LOG="$WORK/logs/train.live.log"
DRV="$WORK/logs/train.driver.log"
rm -f "$LOG" "$DRV"

(
  WORK="$WORK" \
  ENV_NAME="${ENV_NAME:-verl041}" \
  TRAIN_LIVE_LOG="$LOG" \
  MODEL="Qwen/Qwen3-0.6B-MLX-bf16" \
  EXPERIMENT_NAME="Qwen3-0.6B-MLX-bf16-verl-gpu1-fullft" \
  TOTAL_STEPS=50 \
  SAVE_FREQ=50 \
  TEST_FREQ=5 \
  VAL_BEFORE_TRAIN=True \
  RESUME_MODE=disable \
  ROLLOUT_USE_INFERENCE_CHAT_TEMPLATE=True \
  TRAIN_BATCH_SIZE=10 \
  VAL_BATCH_SIZE=5 \
  PPO_MINI_BATCH_SIZE=2 \
  MAX_PROMPT_LENGTH=2048 \
  MAX_RESPONSE_LENGTH=256 \
  MAX_MODEL_LEN=2560 \
  PPO_MAX_TOKEN_LEN_PER_GPU=512 \
  FORWARD_MAX_TOKEN_LEN_PER_GPU=512 \
  LOG_PROB_MAX_TOKEN_LEN_PER_GPU=512 \
  ROLLOUT_MAX_BATCHED_TOKENS=512 \
  ROLLOUT_MAX_SEQS=3 \
  ROLLOUT_N=3 \
  MAX_ASSISTANT_TURNS=3 \
  MAX_TOOL_CALLS_PER_TURN=4 \
  GPU_MEMORY_UTILIZATION=0.35 \
  ROLLOUT_ENABLE_CHUNKED_PREFILL=False \
  ROLLOUT_DISABLE_LOG_STATS=False \
  TRAIN_HEARTBEAT_SEC=10 \
  TRAIN_STDOUT_MODE=off \
  AUTO_START_API=1 \
  bash "$REPO/deploy/v2/train.sh"
) >"$DRV" 2>&1 &

PID=$!
echo "train pid=$PID"
echo "driver log: $DRV"
echo "live log:   $LOG"

while kill -0 "$PID" 2>/dev/null; do
  echo "---- $(date '+%F %T') ----"

  echo "[driver/heartbeat]"
  if [ -f "$DRV" ]; then
    echo "driver_bytes=$(wc -c < "$DRV")"
    tail -n 20 "$DRV" || true
  else
    echo "waiting for $DRV ..."
  fi

  echo "[train/live]"
  if [ -f "$LOG" ]; then
    echo "live_bytes=$(wc -c < "$LOG")"
    tail -n 30 "$LOG" || true
  else
    echo "waiting for $LOG ..."
  fi

  W="$(ls -t /tmp/ray/session_latest/logs/worker-*.err 2>/dev/null | head -n 1 || true)"
  if [ -n "${W:-}" ]; then
    echo "[latest worker err] $W"
    tail -n 8 "$W" || true
  fi

  sleep 10
done

set +e
wait "$PID"
RC=$?
set -e

echo "train rc=$RC"
echo "======== FINAL driver tail ========"
tail -n 120 "$DRV" || true
echo "======== FINAL live tail ========"
tail -n 120 "$LOG" || true
exit "$RC"


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/aae_env.sh
mkdir -p "$WORK/logs"

LATEST_WORKER="$(ls -t /tmp/ray/session_latest/logs/worker-*.err 2>/dev/null | head -n 1 || true)"
OUT="$WORK/logs/debug_bundle_$(date +%Y%m%d_%H%M%S).txt"

{
  echo "=== train.driver.log tail ==="
  tail -n 200 "$WORK/logs/train.driver.log" 2>/dev/null || true
  echo
  echo "=== train.live.log tail ==="
  tail -n 200 "$WORK/logs/train.live.log" 2>/dev/null || true
  echo
  echo "=== latest worker err: $LATEST_WORKER ==="
  [ -n "$LATEST_WORKER" ] && tail -n 200 "$LATEST_WORKER" || true
} > "$OUT"

echo "Saved: $OUT"
tail -n 60 "$OUT" || true
